# Urban Mobility Forecasting

End-to-end Big Data workflow for NYC Yellow Taxi trip records.

## 1. Introduction

This section will present the dataset, source link, project objectives, and target prediction tasks.

## 2. Spark Processing

This section loads, inspects, cleans, transforms, and aggregates the NYC Yellow Taxi trip dataset using both Spark DataFrames and Spark SQL.

### 2.1 Spark Session and Project Paths

Spark is initialized first. The project paths are defined relative to the repository root so the notebook can be run from the `notebooks/` folder without hard-coded absolute paths.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

spark = (
    SparkSession.builder
    .appName("UrbanMobilityForecasting")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

### 2.2 Locate Raw Parquet Files

The project expects multiple monthly Yellow Taxi Parquet files in `data/raw/`, for example `yellow_tripdata_2024-01.parquet`, `yellow_tripdata_2024-02.parquet`, and so on. Spark will read all matching files as one distributed dataset.

In [ ]:
parquet_files = sorted(RAW_DATA_DIR.glob("yellow_tripdata_*.parquet"))

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Number of monthly Parquet files found: {len(parquet_files)}")

for file_path in parquet_files[:12]:
    size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"- {file_path.name}: {size_mb:.2f} MB")

if not parquet_files:
    raise FileNotFoundError(
        "No Yellow Taxi Parquet files were found in data/raw/. "
        "Download monthly files from the NYC TLC Trip Record Data page before running this section."
    )

### 2.3 Load the Dataset with Spark DataFrames

The Parquet files are loaded into a Spark DataFrame. Parquet is columnar, so Spark can efficiently read only the columns needed for later operations.

In [ ]:
trips_df = spark.read.parquet(*[str(path) for path in parquet_files])

print(f"Number of raw rows: {trips_df.count():,}")
trips_df.printSchema()
trips_df.limit(5).toPandas()

### 2.4 Select Useful Columns

Only the columns needed for analysis and modeling are kept. This makes the next transformations easier to understand and reduces unnecessary data movement.

In [ ]:
selected_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
]

missing_columns = [column for column in selected_columns if column not in trips_df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

trips_selected_df = trips_df.select(*selected_columns)
trips_selected_df.limit(5).toPandas()

### 2.5 Initial Data Quality Checks

Before cleaning, the notebook checks missing values and basic numerical ranges. This helps justify the cleaning rules instead of applying them blindly.

In [ ]:
important_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "payment_type",
    "fare_amount",
    "total_amount",
]

missing_summary_df = trips_selected_df.select([
    F.count(F.when(F.col(column).isNull(), column)).alias(column)
    for column in important_columns
])

missing_summary_df.toPandas()

In [ ]:
numeric_summary_df = trips_selected_df.select(
    F.min("passenger_count").alias("min_passenger_count"),
    F.max("passenger_count").alias("max_passenger_count"),
    F.min("trip_distance").alias("min_trip_distance"),
    F.max("trip_distance").alias("max_trip_distance"),
    F.min("fare_amount").alias("min_fare_amount"),
    F.max("fare_amount").alias("max_fare_amount"),
    F.min("total_amount").alias("min_total_amount"),
    F.max("total_amount").alias("max_total_amount"),
)

numeric_summary_df.toPandas()

### 2.6 Clean and Transform the Data

The following transformations create trip duration and temporal features, then remove records that are invalid or unrealistic for modeling.

In [ ]:
trips_features_df = (
    trips_selected_df
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0,
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("is_weekend", F.col("pickup_day_of_week").isin([1, 7]).cast("int"))
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    .withColumn("average_speed_mph", F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0))
)

clean_trips_df = (
    trips_features_df
    .dropna(subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "payment_type",
        "fare_amount",
        "total_amount",
        "trip_duration_minutes",
    ])
    .filter(F.col("trip_duration_minutes") > 0)
    .filter(F.col("trip_duration_minutes") <= 180)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_distance") <= 100)
    .filter(F.col("passenger_count").between(1, 6))
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("total_amount") > 0)
    .filter(F.col("average_speed_mph").between(1, 100))
    .withColumn(
        "trip_distance_bucket",
        F.when(F.col("trip_distance") < 2, "short")
         .when(F.col("trip_distance") < 8, "medium")
         .otherwise("long"),
    )
)

raw_count = trips_selected_df.count()
clean_count = clean_trips_df.count()

print(f"Raw rows: {raw_count:,}")
print(f"Clean rows: {clean_count:,}")
print(f"Rows removed: {raw_count - clean_count:,}")
print(f"Retention rate: {clean_count / raw_count:.2%}")

### 2.7 Cache the Clean Dataset

The cleaned DataFrame is cached because it is reused by multiple analyses and later machine learning stages.

In [ ]:
clean_trips_df = clean_trips_df.cache()
clean_trips_df.count()
clean_trips_df.limit(5).toPandas()

### 2.8 DataFrame Aggregations

These examples use Spark DataFrame operations for grouped analysis over the cleaned dataset.

In [ ]:
trips_by_hour_df = (
    clean_trips_df
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy("pickup_hour")
)

trips_by_hour_df.toPandas()

In [ ]:
payment_type_summary_df = (
    clean_trips_df
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy(F.desc("trip_count"))
)

payment_type_summary_df.toPandas()

In [ ]:
distance_bucket_summary_df = (
    clean_trips_df
    .groupBy("trip_distance_bucket")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
    )
    .orderBy("trip_distance_bucket")
)

distance_bucket_summary_df.toPandas()

### 2.9 Spark SQL Analysis

The same cleaned DataFrame is registered as a temporary SQL view. This satisfies the Spark SQL requirement and makes the analysis readable for people familiar with SQL.

In [ ]:
clean_trips_df.createOrReplaceTempView("taxi_trips")

In [ ]:
spark.sql("""
    SELECT
        pickup_month,
        COUNT(*) AS trip_count,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        ROUND(AVG(total_amount), 2) AS avg_total_amount,
        ROUND(AVG(trip_distance), 2) AS avg_distance
    FROM taxi_trips
    GROUP BY pickup_month
    ORDER BY pickup_month
""").toPandas()

In [ ]:
spark.sql("""
    SELECT
        is_weekend,
        trip_distance_bucket,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_minutes
    FROM taxi_trips
    GROUP BY is_weekend, trip_distance_bucket
    ORDER BY is_weekend, trip_distance_bucket
""").toPandas()

### 2.10 Save the Processed Dataset

The cleaned dataset is written to `data/processed/clean_taxi_trips.parquet`. Later sections can load this processed version instead of repeating the full cleaning pipeline.

In [ ]:
processed_output_path = PROCESSED_DATA_DIR / "clean_taxi_trips.parquet"

(
    clean_trips_df
    .write
    .mode("overwrite")
    .parquet(str(processed_output_path))
)

print(f"Processed dataset saved to: {processed_output_path}")

## 3. Spark MLlib Models

This section will include at least two Spark MLlib methods for regression and/or classification.

## 4. Data Pipeline

This section will define at least one Spark ML Pipeline.

## 5. UDF and Hyperparameter Tuning

This section will include a user-defined function and model tuning with a parameter grid.

## 6. TensorFlow Deep Learning

This section will train and evaluate a TensorFlow model on selected processed features.

## 7. Spark Streaming

This section will simulate streaming input and apply online inference or streaming analytics.